In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# sinc function
def sinc(x):
    small_mask = np.abs(x) > 1e-10
    y = np.zeros_like(x)
    y[small_mask == 1] = np.sin(x[small_mask == 1]) / x[small_mask == 1]
    y[small_mask == 0] = 1
    return y

## E-Field & Directivity for a Single Facet

In [ ]:
def facet_rad(theta, phi, l, lam):
    sinc1 = ((np.pi * l) / lam) * np.sin(theta) * np.cos(phi)
    sinc2 = ((np.pi * l) / lam) * np.sin(theta) * np.sin(phi)
    D0 = (4 * np.pi * l**2) / lam**2
    return D0 * ((sinc(sinc1) * sinc(sinc2))**2)

def facet_E(theta, phi, l, lam, R, E_0=1):
    sinc1 = ((np.pi * l) / lam) * np.sin(theta) * np.cos(phi)
    sinc2 = ((np.pi * l) / lam) * np.sin(theta) * np.sin(phi)
    k = (2 * np.pi) / lam
    const = ((1j * E_0 * l * l) / lam) * (np.exp(-1j * k * R) / R)
    return const * sinc(sinc1) * sinc(sinc2)

In [ ]:
c = 299792458
f = 60e6
lam = c / f
l_1 = 50
l_2 = 500

alt = 25e3

r2d = 180 / np.pi

thetas = np.linspace(-1*np.pi/2, np.pi/2, 1000)
Ds_1 = facet_rad(thetas, 0, l_1, lam)
Ds_2 = facet_rad(thetas, 0, l_2, lam)
E_1  = facet_E(thetas, 0, l_1, lam, alt)
E_2  = facet_E(thetas, 0, l_2, lam, alt)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for a, l, D in zip(ax, (l_1, l_2), (Ds_1, Ds_2)):
    a.plot(thetas*r2d, 10*np.log10(D), color="red", linewidth=1)
    a.set_xlabel("Theta [deg]"); 
    a.set_xlim(-90, 90); a.set_ylim(-20, 53)
    a.set_title(f"Directivity for a facet of size {l/lam:.0f} lambda", fontsize=10)
    a.grid(True, axis='y', alpha=0.6, linestyle=":")
ax[0].set_ylabel("Normalized Directivity [dB]")
plt.subplots_adjust(wspace=0.02)
plt.savefig("IndividualFacetDirectivity.png")
plt.show()

In [ ]:
MD = np.max(Ds_2) / np.max(Ds_1)
print(f"Peak directivity ratio of {MD:.1f} or {10*np.log10(MD):.1f} dB") 

In [ ]:
def HPBW(D, angles):
    D_norm = D / np.max(D)
    HPBW_mask = D_norm > 0.5
    d_deg = (180 / np.pi) * (angles[1] - angles[0])
    return np.sum(HPBW_mask) * d_deg

In [ ]:
print(f"HPBW for {l_1/lam:.0f} lam is {HPBW(Ds_1, thetas):.2f} deg")
print(f"HPBW for {l_2/lam:.0f} lam is {HPBW(Ds_2, thetas):.2f} deg")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4), sharey=False)
for a, l, E in zip(ax, (l_1, l_2), (E_1, E_2)):
    a.plot(thetas*r2d, np.real(E), color="red", linewidth=1, label="real")
    a.plot(thetas*r2d, np.imag(E), color="blue", linewidth=1, label="imag")
    a.plot(thetas*r2d, np.abs(E), color="black", linewidth=1, label="abs")
    a.set_xlabel("Theta [deg]"); 
    a.set_xlim(-90, 90)
    a.set_title(f"E field at {alt/1e3} km a facet of size {l/lam:.0f} lam", fontsize=10)
    a.grid(True, axis='y', alpha=0.6, linestyle=":")
    a.legend()
ax[0].set_ylabel("E-Field [V/m]")
plt.subplots_adjust(wspace=0.1)
plt.savefig("IndividualFacetEField.png")
plt.show()

## Array Factor

In [ ]:
def psi_AF(lam, d, theta, beta=0):
    k = (2 * np.pi) / lam
    return k * d * np.sin(theta) + beta

In [ ]:
def array_factor_1D(lam, d, theta, N, beta=0):
    psi = psi_AF(lam, d, theta, beta=beta)
    AF = np.zeros(len(theta), dtype=np.complex128)
    for i in range(1, N+1):
        AF += np.exp(1j*(i-1)*psi)
    return AF

# For an array of $l_1$ facets, how well do they emulate a $l_2$ facet?

In [ ]:
dist = l_1
N    = int(l_2/l_1)
AF = array_factor_1D(lam, dist, thetas, N)
Ds_1_arr = Ds_1 * (AF**2)
E_1_arr  = E_1 * AF

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4), sharey=False)
ax[0].plot(thetas*r2d, np.real(AF), color="red", label="real")
ax[0].plot(thetas*r2d, np.imag(AF), color="blue", label="imag")
ax[0].set_title("Array Factor Components")
for a in ax: a.set_ylabel("Array Factor [.]")
for a in ax: a.set_xlabel("Theta [deg]")
ax[1].set_title("Array Factor Squared")
ax[1].plot(thetas*r2d, AF**2, color="black")
plt.savefig("ArrayFactor.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for a, l, D in zip(ax, (l_1, l_2), (Ds_1_arr, Ds_2)):
    a.plot(thetas*r2d, 10*np.log10(D), color="red", linewidth=1)
    a.set_xlabel("Theta [deg]"); 
    a.set_xlim(-90, 90); a.set_ylim(-20, 53)
    a.grid(True, axis='y', alpha=0.6, linestyle=":")
ax[0].set_ylabel("Normalized Directivity [dB]")
ax[0].set_title(f"{N}x{N} array of {l_1/lam:.0f} lam facets", fontsize=10)
ax[1].set_title(f"Single facet of size {l/lam:.0f} lambda", fontsize=10)
plt.subplots_adjust(wspace=0.02)
plt.savefig("FacetArrayDirectivity.png")
plt.show()

In [ ]:
print(f"HPBW for {l_1/lam:.0f} lam is {HPBW(Ds_1_arr, thetas):.2f} deg")
print(f"HPBW for {l_2/lam:.0f} lam is {HPBW(Ds_2, thetas):.2f} deg")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4), sharey=False)
for a, l, E in zip(ax, (l_1, l_2), (E_1_arr, E_2)):
    a.plot(thetas*r2d, np.real(E), color="red", linewidth=1, label="real")
    a.plot(thetas*r2d, np.imag(E), color="blue", linewidth=1, label="imag")
    a.plot(thetas*r2d, np.abs(E), color="black", linewidth=1, label="abs")
    a.set_xlabel("Theta [deg]"); 
    a.set_xlim(-90, 90)
    a.grid(True, axis='y', alpha=0.6, linestyle=":")
    a.legend()
ax[0].set_ylabel("E-Field [V/m]")
ax[0].set_title(f"{N}x{N} array of {l_1/lam:.0f} lam facets", fontsize=10)
ax[1].set_title(f"Single facet of size {l/lam:.0f} lambda", fontsize=10)
plt.subplots_adjust(wspace=0.1)
plt.savefig("FacetArrayEField.png")
plt.show()